In [ ]:
import pandas as pd
import cv2
import numpy as np
import math
import os
import glob
import time
import json
import re
import shutil

from PIL import Image
from pathlib import Path
from google.colab.patches import cv2_imshow
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

from google import genai
from collections import defaultdict

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

In [ ]:
!unzip /content/drive/MyDrive/SUPER_AI/simple_data.zip

In [ ]:
def deskew_image(image_path):
  img = cv2.imread(image_path)
  if img is None: raise ValueError(f'ไม่พบไฟล์: {image_path}')

  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  denoised = cv2.fastNlMeansDenoising(gray, None, h=20, templateWindowSize=7, searchWindowSize=21)

  #adaptiveThreshold (การตัดขาว-ดำ)
  thresh = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 5)

  h_img, w_img = gray.shape
  h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(1, w_img // 20), 1))

  h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel, iterations=2)

  angle = 0.0
  lines = cv2.HoughLinesP(h_lines, 1, np.pi/180, threshold=50, minLineLength=w_img//10, maxLineGap=20)
  if lines is not None:
      angles = [math.degrees(math.atan2(line[0][3] - line[0][1], line[0][2] - line[0][0])) for line in lines]
      angles = [ang for ang in angles if abs(ang) < 45]
      if angles: angle = np.median(angles)

  M = cv2.getRotationMatrix2D((w_img // 2, h_img // 2), angle, 1.0)
  rotated = cv2.warpAffine(img, M, (w_img, h_img), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=(255,255,255))

  return rotated, angle

In [ ]:
def enhance_sharpness(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()

    # ลบเงาและปรับแสงให้สม่ำเสมอ
    blur = cv2.GaussianBlur(gray, (0,0), sigmaX=33, sigmaY=33)
    divided = cv2.divide(gray, blur, scale=255)

    # ดึงคอนทราสต์ให้สุด
    stretched = cv2.normalize(divided, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

    # Unsharp Masking ให้ตัวหนังสือคมกริบ
    gaussian = cv2.GaussianBlur(stretched, (0, 0), 3.0)
    sharpened = cv2.addWeighted(stretched, 2.5, gaussian, -1.5, 0)

    return cv2.cvtColor(sharpened, cv2.COLOR_GRAY2BGR)

In [ ]:
def remove_table_lines(image):
    """ ลบเส้นตารางแนวตั้งและแนวนอนออกจากภาพ โดยไม่ให้กระทบตัวหนังสือ """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 5)

    # หาเส้นแนวตั้ง kernel 80 pixel
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 30))
    v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel, iterations=2)

    # หาเส้นแนวนอน
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (100, 1))
    h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel, iterations=2)

    # รวมเส้นตั้งและเส้นนอนเข้าด้วยกัน
    table_mask = cv2.add(v_lines, h_lines)

    # ขยายขนาดเส้นเล็กน้อยให้ลบได้สะอาดขึ้น
    table_mask = cv2.dilate(table_mask, np.ones((5,5), np.uint8), iterations=1)

    # ลบเส้นออกจากภาพต้นฉบับ (แทนที่ตำแหน่งเส้นด้วยสีขาว [255, 255, 255])
    result = image.copy()
    result[table_mask == 255] = [255, 255, 255]

    return result

In [ ]:
def process_single_image(file_path, output_folder, output_folder_no_chunk):
    filename = os.path.basename(file_path)
    base_name = os.path.splitext(filename)[0]

    try:
        deskewed_img, angle = deskew_image(file_path)

        enhanced_full_img = enhance_sharpness(deskewed_img)

        # เซฟแบบไม่หั่นลงใน output_folder_no_chunk
        os.makedirs(output_folder_no_chunk, exist_ok=True)
        filename_no_chunk = os.path.join(output_folder_no_chunk, f"{base_name}.png")
        cv2.imwrite(filename_no_chunk, enhanced_full_img)
        gray = cv2.cvtColor(deskewed_img, cv2.COLOR_BGR2GRAY)

        brightness = np.mean(gray)
        adaptive_c = 5 if brightness > 120 else 2

        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, adaptive_c)

        h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (gray.shape[1] // 20, 1))
        h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel, iterations=2)
        h_lines = cv2.dilate(h_lines, cv2.getStructuringElement(cv2.MORPH_RECT, (20, 1)), iterations=2)

        contours, _ = cv2.findContours(h_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        y_coords = sorted([cv2.boundingRect(c)[1] for c in contours if cv2.boundingRect(c)[2] > gray.shape[1] * 0.2])

        merged_y = []
        for y in y_coords:
            if not merged_y or y - merged_y[-1] > 15:
                merged_y.append(y)

        processed_rows = []

        for i in range(len(merged_y) - 1):
            y_start, y_end = merged_y[i], merged_y[i+1]
            row_height = y_end - y_start

            if 20 < row_height < 500:
                safe_y_start = max(0, y_start - 10)
                safe_y_end = min(deskewed_img.shape[0], y_end + 5)

                row_img = deskewed_img[safe_y_start:safe_y_end, 0:deskewed_img.shape[1]]

                gray_row = cv2.cvtColor(row_img, cv2.COLOR_BGR2GRAY)
                _, std_dev = cv2.meanStdDev(gray_row)
                if std_dev[0][0] < 5.0:
                    continue

                enhanced_row = enhance_sharpness(row_img)

                row_pad = 60
                padded_row = cv2.copyMakeBorder(
                    enhanced_row,
                    row_pad, row_pad, row_pad, row_pad,
                    cv2.BORDER_CONSTANT, value=[255, 255, 255]
                )

                processed_rows.append(padded_row)

        total_rows = len(processed_rows)
        saved_files = []

        if total_rows > 0:
            ROWS_PER_CHUNK = 10

            for chunk_idx in range(0, total_rows, ROWS_PER_CHUNK):
                chunk_rows = processed_rows[chunk_idx : chunk_idx + ROWS_PER_CHUNK]

                stitched_chunk = cv2.vconcat(chunk_rows)

                cleaned_chunk = remove_table_lines(stitched_chunk)

                global_pad = 120
                final_padded_img = cv2.copyMakeBorder(
                    cleaned_chunk,
                    global_pad, global_pad, global_pad, global_pad,
                    cv2.BORDER_CONSTANT, value=[255, 255, 255]
                )

                part_number = (chunk_idx // ROWS_PER_CHUNK) + 1

                os.makedirs(output_folder, exist_ok=True)

                filename_out = os.path.join(output_folder, f"{base_name}_chunk{part_number}.png")
                cv2.imwrite(filename_out, final_padded_img)
                saved_files.append(filename_out)

        return True, filename, angle, total_rows, saved_files

    except Exception as ex:
        return False, filename, 0.0, 0, str(ex), None

In [ ]:
def preProcessing(input_folder, output_folder, output_folder_no_chunk):
    start_time = time.time()
    log_file = os.path.join(output_folder, "processed_log.txt")
    os.makedirs(output_folder, exist_ok=True)

    try:
        completed_originals = set()
        if os.path.exists(log_file):
            with open(log_file, "r") as f:
                completed_originals = set(line.strip() for line in f)

        done_files = os.listdir(output_folder)
        for f in done_files:
            if "_chunk" in f:
                completed_originals.add(f.split("_chunk")[0])

        all_image_paths = glob.glob(os.path.join(input_folder, "*.png"))
        to_process = []
        for file_path in all_image_paths:
            base_name = os.path.splitext(os.path.basename(file_path))[0]
            if base_name not in completed_originals:
                to_process.append(file_path)

        files_to_run = len(to_process)
        if files_to_run == 0:
            print("✅ ทุกไฟล์ประมวลผลเสร็จหมดแล้ว!")
            return

        print(f"🚀 เริ่มประมวลผล {files_to_run} ไฟล์ที่เหลือ...")

        success_count = 0
        workers = min(os.cpu_count(), 5)

        with ProcessPoolExecutor(max_workers=workers) as executor:
            futures = {executor.submit(process_single_image, f, output_folder, output_folder_no_chunk): f for f in to_process}

            with open(log_file, "a") as log:
                for future in as_completed(futures):
                    success, filename, angle, rows, result_data = future.result()

                    if success:
                        success_count += 1
                        log.write(f"{filename}\n")
                        log.flush()

                        print(f"[{success_count}/{files_to_run}] {filename} -> {rows} แถว")
                    else:
                        print(f"❌ ล้มเหลว: {filename} - {result_data}")

        end_time = time.time()
        print("\n" + "="*50)
        print(f"🎉 เสร็จสิ้น! ใช้เวลาทั้งหมด: {end_time - start_time:.2f} วินาที")
        print(f"📁 บันทึกไฟล์ที่: {output_folder}")
        print("="*50)

    except Exception as e:
        print(f"เกิดข้อผิดพลาดรุนแรง: {e}")

In [ ]:
def process_model(image_path):
    try:
        file_name = Path(image_path).stem
        print(f"กำลังเริ่มประมวลผล: {file_name}")

        img = Image.open(image_path)

        client = genai.Client(api_key=api_key_gemini)

        attempt = 0
        success = False
        base_delay = 2

        while not success:
            attempt += 1
            try:
                if attempt > 1:
                    print(f"กำลังลองใหม่ครั้งที่ {attempt} สำหรับ: {file_name}...")

                response = client.models.generate_content(
                    model="gemini-3.1-flash-lite-preview",
                    contents=[my_prompt, img],
                    config={
                        "temperature": 0.1,
                        "top_p": 0.6,
                        "max_output_tokens": 4000,
                        "response_mime_type": "application/json"
                    }
                )

                if response and response.text:
                    raw_text = response.text.strip()

                    if raw_text.startswith("```json"):
                        raw_text = raw_text[7:-3].strip()
                    elif raw_text.startswith("```"):
                        raw_text = raw_text[3:-3].strip()

                    try:
                        parsed_json = json.loads(raw_text)
                        save_path = os.path.join(output_json_after_ocr, f"{file_name}.json")

                        with open(save_path, "w", encoding="utf-8") as f:
                            json.dump(parsed_json, f, ensure_ascii=False, indent=4)

                        print(f"✅ สำเร็จ: {save_path}")

                        success = True
                        time.sleep(1)

                    except json.JSONDecodeError:
                        print(f"{file_name} (รอบ {attempt}): API ส่ง JSON ไม่ถูกต้อง!")

                        # คำนวณเวลารอ หน่วงเวลาเพิ่มขึ้นเรื่อยๆ แต่สูงสุดไม่เกิน 30 วินาที
                        wait_time = min(base_delay * attempt, 30)
                        print(f"รอ {wait_time} วินาทีก่อนลองใหม่...")
                        time.sleep(wait_time)

                else:
                    print(f"{file_name}: ไม่ได้รับข้อความจาก API")
                    wait_time = min(base_delay * attempt, 30)
                    time.sleep(wait_time)

            except Exception as e:
                print(f"พยายามครั้งที่ {attempt} ล้มเหลว ({file_name}): {e}")
                wait_time = min(base_delay * attempt, 30)
                time.sleep(wait_time)

    except Exception as e:
        print(f"Error ประมวลผลไฟล์ {image_path} ขั้นต้น: {str(e)}")

In [ ]:
def model_training(input_folder_after_manual, output_json_after_ocr):
  os.makedirs(output_json_after_ocr, exist_ok=True)

  all_images = glob.glob(os.path.join(input_folder_after_manual, "*.png"))

  if not all_images:
      print(f"⚠️ ไม่พบไฟล์รูปภาพ .png ในโฟลเดอร์ {input_folder_after_manual}")
      return

  existing_jsons = glob.glob(os.path.join(output_json_after_ocr, "*.json"))
  processed_names = {Path(f).stem for f in existing_jsons}

  image_list = [img for img in all_images if Path(img).stem not in processed_names]

  total_files = len(all_images)
  already_done = len(processed_names)
  to_process = len(image_list)

  print(f"📁 พบไฟล์ต้นฉบับทั้งหมด: {total_files} ไฟล์")
  print(f"✅ ทำเสร็จไปแล้ว: {already_done} ไฟล์")
  print(f"🚀 เหลือที่ต้องทำอีก: {to_process} ไฟล์")

  if not image_list:
      print("\n🎉 เย้! ประมวลผลครบทุกไฟล์เรียบร้อยแล้ว ไม่มีอะไรต้องทำเพิ่มครับ")
      return

  print(f"\nเริ่มรันแบบ Parallel (Workers={max_workers_training})...")

  with ThreadPoolExecutor(max_workers=max_workers_training) as executor:
      executor.map(process_model, image_list)

  print("\n--- ประมวลผลชุดนี้เสร็จสิ้น ---")

In [ ]:
def split_dataset(input_folder, output_folder, num_splits=6):

    os.makedirs(output_folder, exist_ok=True)

    files = glob.glob(os.path.join(input_folder, "*.png"))
    files.sort()

    total_files = len(files)
    if total_files == 0:
        print(f"❌ ไม่พบไฟล์ .png ในโฟลเดอร์ '{input_folder}'")
        return

    print(f"📦 พบไฟล์ทั้งหมด {total_files} ไฟล์ กำลังแบ่งเป็น {num_splits} ส่วน...")

    chunk_size = math.ceil(total_files / num_splits)

    for i in range(num_splits):
        split_folder = os.path.join(output_folder, f"part_{i+1}")
        os.makedirs(split_folder, exist_ok=True)

        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, total_files)
        chunk_files = files[start_idx:end_idx]

        print(f"📁 Part {i+1}: กำลังคัดลอก {len(chunk_files)} ไฟล์ลงใน '{split_folder}'...")

        for file_path in chunk_files:
            file_name = os.path.basename(file_path)
            dest_path = os.path.join(split_folder, file_name)
            shutil.copy2(file_path, dest_path)

    print("\n✅ แบ่งไฟล์เสร็จสมบูรณ์!")

In [ ]:
def update_submission_scores_by_index(template_csv_path, input_json_path, output_csv_path):
    if not os.path.exists(template_csv_path):
        print(f"❌ ไม่พบไฟล์ต้นแบบ: {template_csv_path}")
        return

    print("⏳ กำลังโหลดไฟล์ CSV ต้นแบบ...")
    df = pd.read_csv(template_csv_path)

    if 'id' not in df.columns or 'votes' not in df.columns:
        print("❌ ไฟล์ CSV ต้องมีคอลัมน์ 'id' และ 'votes'")
        return

    df.set_index('id', inplace=True)
    df['votes'] = pd.to_numeric(df['votes'], errors='coerce').fillna(0).astype(int)

    json_files = glob.glob(os.path.join(input_json_path, '*.json'))

    if not json_files:
        print(f"⚠️ ไม่พบไฟล์ .json ในโฟลเดอร์ {input_json_path}")
        return

    print(f"🔍 พบ JSON {len(json_files)} ไฟล์ เริ่มอัปเดตตามลำดับ...")

    for json_file in json_files:
        base_name = os.path.basename(json_file)
        raw_doc_id = os.path.splitext(base_name)[0]

        doc_id = re.sub(r'_page.*$', '', raw_doc_id, flags=re.IGNORECASE).strip()

        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            print(f"⚠️ อ่านไฟล์ {base_name} ไม่ได้: {e}")
            continue

        for i, item in enumerate(data, 1):
            score_str = item.get("คะแนนตัวเลข", "0").strip()

            try:
                score = int(score_str.replace(',', ''))
            except ValueError:
                score = 0

            target_id = f"{doc_id}_{i}"

            if target_id in df.index:
                df.at[target_id, 'votes'] += score
            else:
                # กรณีฉุกเฉิน: ถ้าหา ID ไม่เจอ
                print(f"ไม่พบ ID: {target_id} ใน CSV")
                pass

    df.reset_index().to_csv(output_csv_path, index=False, encoding='utf-8')
    print(f"✅ สำเร็จ! บันทึกไปที่: {output_csv_path}")

In [ ]:
def merged_json(input_dir, output_dir):
  os.makedirs(output_dir, exist_ok=True)

  pattern = re.compile(r'^(.+?)(?:_page(\d+))?(?:_chunk(\d+))?\.json$')

  grouped_files = defaultdict(list)

  for filepath in glob.glob(os.path.join(input_dir, '*.json')):
      filename = os.path.basename(filepath)
      match = pattern.match(filename)

      if match:
          base_name = match.group(1)

          page_num = int(match.group(2)) if match.group(2) else 1

          chunk_num = int(match.group(3)) if match.group(3) else 0

          grouped_files[base_name].append((page_num, chunk_num, filepath))
      else:
          print(f"ข้ามไฟล์ที่ไม่เข้าเงื่อนไข: {filename}")

  for base_name, file_infos in grouped_files.items():

      file_infos.sort(key=lambda x: (x[0], x[1]))

      merged_data = {
          "results": [],
          "page_summary": None
      }

      print(f"\nกำลังรวมกลุ่ม: {base_name}")

      for page_num, chunk_num, file_path in file_infos:
          print(f"  ┣ อ่านไฟล์: {os.path.basename(file_path)}")

          with open(file_path, 'r', encoding='utf-8') as f:
              try:
                  data = json.load(f)

                  if "results" in data and isinstance(data["results"], list):
                      merged_data["results"].extend(data["results"])

                  if data.get("page_summary") is not None:
                      merged_data["page_summary"] = data["page_summary"]

              except json.JSONDecodeError:
                  print(f"  [Error] อ่านไฟล์พัง: {os.path.basename(file_path)}")

      output_filepath = os.path.join(output_dir, f"{base_name}.json")
      with open(output_filepath, 'w', encoding='utf-8') as f:
          json.dump(merged_data, f, ensure_ascii=False, indent=4)

      print(f"✅ บันทึก {base_name}.json สำเร็จ (จำนวน {len(merged_data['results'])} รายการ)")

  print("\n🎉 จัดเรียงและรวมไฟล์เสร็จสมบูรณ์!")

In [ ]:
def convert_numeric_to_int(text_val):
    """แปลงตัวเลข (ทั้งอารบิกและไทย) ที่มีลูกน้ำ ให้เป็น integer"""
    if not text_val or str(text_val).strip() in ["", "-", "ไม่มี"]:
        return 0

    text_val = str(text_val).replace(',', '').strip()

    thai_to_arabic = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
    cleaned = text_val.translate(thai_to_arabic)

    try:
        return int(cleaned)
    except ValueError:
        return 0

In [ ]:
def convert_thai_text_to_int(text_val):
    """แปลงคำอ่านภาษาไทย (เช่น 'หนึ่งร้อยห้าสิบสี่') ให้เป็น integer"""
    if not text_val or str(text_val).strip() == "":
        return 0

    text_val = str(text_val).replace(" ", "")
    if text_val == "ศูนย์":
        return 0

    values = {
        'ศูนย์': 0, 'หนึ่ง': 1, 'เอ็ด': 1, 'สอง': 2, 'ยี่': 2,
        'สาม': 3, 'สี่': 4, 'ห้า': 5, 'หก': 6, 'เจ็ด': 7, 'แปด': 8, 'เก้า': 9
    }
    places = {'สิบ': 10, 'ร้อย': 100, 'พัน': 1000, 'หมื่น': 10000, 'แสน': 100000, 'ล้าน': 1000000}

    total = 0
    current = 0

    pattern = re.compile('ล้าน|แสน|หมื่น|พัน|ร้อย|สิบ|ศูนย์|หนึ่ง|เอ็ด|สอง|ยี่|สาม|สี่|ห้า|หก|เจ็ด|แปด|เก้า')
    tokens = pattern.findall(text_val)

    for token in tokens:
        if token in values:
            current = values[token]
        elif token in places:
            if current == 0 and token == 'สิบ':
                current = 1
            total += current * places[token]
            current = 0

    total += current
    return total

In [ ]:
def validate_and_categorize_results(input_dir):
    pass_dir = os.path.join(input_dir, "passed_complete")
    review_dir = os.path.join(input_dir, "needs_review")

    os.makedirs(pass_dir, exist_ok=True)
    os.makedirs(review_dir, exist_ok=True)

    files = [f for f in glob.glob(os.path.join(input_dir, '*.json')) if os.path.isfile(f)]

    needs_review_list = []

    # รวมการเก็บสถิติทั้งแบบรวมและแบบแยกย่อย
    report = {
        "passed_total": 0,
        "needs_review": 0,
        "passed_num_match": 0,
        "passed_text_match": 0,
        "passed_no_summary_match": 0
    }

    for file_path in files:
        filename = os.path.basename(file_path)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        results = data.get("results", [])
        summary = data.get("page_summary")

        calc_sum_num = sum(convert_numeric_to_int(item.get("score_numeric")) for item in results)
        calc_sum_text = sum(convert_thai_text_to_int(item.get("score_text")) for item in results)

        is_passed = False
        reason = ""

        if summary:
            target_sum_num = convert_numeric_to_int(summary.get("total_score_numeric"))
            target_sum_text = convert_thai_text_to_int(summary.get("total_score_text"))

            if calc_sum_num == target_sum_num:
                is_passed = True
                reason = "ผลรวมตัวเลขตรง"
                report["passed_num_match"] += 1
            elif calc_sum_text == target_sum_text:
                is_passed = True
                reason = "ผลรวมตัวอักษรตรง"
                report["passed_text_match"] += 1
            else:
                reason = "ไม่ตรงทั้งคู่"
        else:
            if calc_sum_num == calc_sum_text and calc_sum_num > 0:
                is_passed = True
                reason = "ไม่มีสรุป แต่ตรงกันเอง"
                report["passed_no_summary_match"] += 1
            else:
                reason = "ไม่มีสรุป และขัดแย้งกัน"

        data["validation"] = {"status": "PASSED" if is_passed else "NEEDS_REVIEW", "reason": reason}

        target_dir = pass_dir if is_passed else review_dir
        target_path = os.path.join(target_dir, filename)

        with open(target_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

        os.remove(file_path)

        if is_passed:
            report["passed_total"] += 1
        else:
            report["needs_review"] += 1
            needs_review_list.append(filename)

    print("=== สรุปผลการตรวจสอบ (Validation Report) ===")
    print(f"✅ ครบถ้วน (Passed Total): {report['passed_total']} ไฟล์ (ดูที่โฟลเดอร์ passed_complete)")
    print(f"   ┣ ผ่านเพราะ 'ผลรวมตัวเลข' ตรงกับหน้าสรุป: {report['passed_num_match']} ไฟล์")
    print(f"   ┣ ผ่านเพราะ 'ผลรวมตัวอักษร' ตรงกับหน้าสรุป (ตัวเลขไม่ตรง): {report['passed_text_match']} ไฟล์")
    print(f"   ┗ ผ่านเพราะ 'ตัวเลขและตัวอักษร' ในตารางตรงกันเอง (ไม่มีหน้าสรุป): {report['passed_no_summary_match']} ไฟล์")
    print(f"⚠️ ต้องตรวจเช็ค (Needs Review): {report['needs_review']} ไฟล์")

    return needs_review_list

In [ ]:
def generate_final_score_json(input_dir, for_check_list, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    all_files = glob.glob(os.path.join(input_dir, '*.json'))

    all_files.extend(glob.glob(os.path.join(input_dir, 'passed_complete', '*.json')))
    all_files.extend(glob.glob(os.path.join(input_dir, 'needs_review', '*.json')))

    all_files = list(set(all_files))

    processed_count = 0

    for file_path in all_files:
        filename = os.path.basename(file_path)

        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        final_results = []

        is_needs_review = filename in for_check_list

        for item in data.get("results", []):
            if is_needs_review:
                score = convert_thai_text_to_int(item.get("score_text"))
            else:
                score = convert_numeric_to_int(item.get("score_numeric"))

            final_results.append({
                "คะแนน": score
            })

        out_path = os.path.join(output_dir, filename)
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(final_results, f, ensure_ascii=False, indent=4)

        processed_count += 1

    print(f"\n🎉 สร้างไฟล์ Final Score จำนวน {processed_count} ไฟล์ เสร็จสิ้น!")
    print(f"📂 ไฟล์ผลลัพธ์อยู่ที่โฟลเดอร์: {output_dir}")

In [ ]:
def merge_scores_to_submission(json_dir, submission_csv_path, output_csv_path):
    print(f"⏳ กำลังโหลดไฟล์ Submission Template: {submission_csv_path}...")

    df = pd.read_csv(submission_csv_path)
    df.set_index('id', inplace=True)

    json_files = glob.glob(os.path.join(json_dir, '*.json'))

    processed_files = 0
    updated_rows = 0
    missing_ids = []

    print(f"🔄 กำลังประมวลผลไฟล์ JSON จำนวน {len(json_files)} ไฟล์...")

    for file_path in json_files:
        basename = os.path.splitext(os.path.basename(file_path))[0]

        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)

                for idx, item in enumerate(data):
                    candidate_no = idx + 1

                    row_id = f"{basename}_{candidate_no}"
                    score = item.get("คะแนน", 0)

                    if row_id in df.index:
                        df.at[row_id, 'votes'] = score
                        updated_rows += 1
                    else:
                        missing_ids.append(row_id)

                processed_files += 1

            except Exception as e:
                print(f"  ❌ เกิดข้อผิดพลาดในการอ่านไฟล์ {basename}: {e}")

    df.reset_index(inplace=True)
    df.to_csv(output_csv_path, index=False, encoding='utf-8')

    print("\n=========================================")
    print(f"🎉 อัปเดตข้อมูลลงไฟล์ Submission สำเร็จ!")
    print(f"📄 ประมวลผลไฟล์ JSON ไปทั้งสิ้น: {processed_files} ไฟล์")
    print(f"✅ อัปเดตคะแนน (Votes) ไปทั้งหมด: {updated_rows} รายการ")
    print(f"💾 บันทึกไฟล์พร้อมส่งที่: {output_csv_path}")
    print("=========================================")

    if missing_ids:
        print(f"⚠️ พบ ID จำนวน {len(missing_ids)} รายการที่ไม่มีในไฟล์ CSV เช่น: {missing_ids[:5]}...")

In [ ]:
def copy_images_for_recheck(basename_list, input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    copied_count = 0
    not_found_list = []
    
    print(f"🔄 กำลังค้นหาและคัดลอกรูปภาพจำนวน {len(basename_list)} รายการ...")
    
    for item in basename_list:
        basename = os.path.splitext(item)[0] 
        
        search_pattern = os.path.join(input_dir, f"{basename}.*")
        matched_files = glob.glob(search_pattern)
        
        valid_extensions = ('.png')
        image_files = [f for f in matched_files if f.lower().endswith(valid_extensions)]
        
        if image_files:
            source_file = image_files[0]
            filename = os.path.basename(source_file)
            dest_file = os.path.join(output_dir, filename)
            
            shutil.copy2(source_file, dest_file)
            copied_count += 1
        else:
            not_found_list.append(basename)
            print(f"  ❌ ไม่พบรูปภาพของ: {basename}")

    # สรุปผลลัพธ์
    print("\n=========================================")
    print(f"✅ คัดลอกสำเร็จ: {copied_count} ไฟล์")
    print(f"📂 รูปภาพถูกเตรียมพร้อมไว้ที่: {output_dir}")
    if not_found_list:
        print(f"⚠️ หาไม่พบ: {len(not_found_list)} ไฟล์ (ไม่มีไฟล์นี้ใน {input_dir})")
    print("=========================================")

In [ ]:
def resolve_and_generate_final_scores(input1_dir, input2_dir, output_dir):
    """
    นำผล OCR รอบแรก (input1) มาเทียบกับผล OCR รอบแก้ตัว (input2)
    - ถ้ามีตัวเลขหรือตัวอักษรตรงกัน -> ยึดค่าที่ตรงกันเป็นหลัก
    - ถ้าไม่ตรงกันเลยสักจุด -> ยึดค่า 'ตัวอักษร' ของ input1
    """
    os.makedirs(output_dir, exist_ok=True)
    
    rechecked_files = glob.glob(os.path.join(input2_dir, '*.json'))
    
    processed_count = 0
    resolved_match = 0
    resolved_fallback = 0
    
    print(f"🔄 กำลังเปรียบเทียบข้อมูลจำนวน {len(rechecked_files)} ไฟล์...")

    for file2_path in rechecked_files:
        filename = os.path.basename(file2_path)
        file1_path = os.path.join(input1_dir, filename)
        
        if not os.path.exists(file1_path):
            print(f"  ⚠️ ข้ามไฟล์ {filename} (ไม่พบไฟล์ต้นฉบับใน {input1_dir})")
            continue
            
        with open(file1_path, 'r', encoding='utf-8') as f1, \
             open(file2_path, 'r', encoding='utf-8') as f2:
            data1 = json.load(f1)
            data2 = json.load(f2)
            
        results1 = data1.get("results", [])
        results2 = data2.get("results", [])
        
        final_results = []
        
        for item1, item2 in zip(results1, results2):
            num1 = convert_numeric_to_int(item1.get("score_numeric"))
            txt1 = convert_thai_text_to_int(item1.get("score_text"))
            
            num2 = convert_numeric_to_int(item2.get("score_numeric"))
            txt2 = convert_thai_text_to_int(item2.get("score_text"))
            
            score = 0
            
            if num2 == num1:             # เลขรอบ 2 ตรงกับ เลขรอบ 1
                score = num2
                resolved_match += 1
            elif txt2 == txt1:           # อักษรรอบ 2 ตรงกับ อักษรรอบ 1
                score = txt2
                resolved_match += 1
            elif num2 == txt1:           # เลขรอบ 2 ตรงกับ อักษรรอบ 1
                score = num2
                resolved_match += 1
            elif txt2 == num1:           # อักษรรอบ 2 ตรงกับ เลขรอบ 1
                score = txt2
                resolved_match += 1
            elif num1 == txt1:           # ในรอบแรกมันตรงกันอยู่แล้ว
                score = num1
                resolved_match += 1
            else:
                score = txt1
                resolved_fallback += 1
                
            final_results.append({
                "คะแนน": score
            })
            
        out_path = os.path.join(output_dir, filename)
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(final_results, f, ensure_ascii=False, indent=4)
            
        processed_count += 1

    print("\n=========================================")
    print(f"🎉 เปรียบเทียบและสร้างไฟล์ Final Score สำเร็จ {processed_count} ไฟล์!")
    print(f"   ┣ ได้ข้อสรุปจากการ 'ตรงกัน (Match)': {resolved_match} รายการ")
    print(f"   ┗ ได้ข้อสรุปจากการ 'ยึดอักษร Input 1 (Fallback)': {resolved_fallback} รายการ")
    print(f"📂 ไฟล์พร้อมใช้งานอยู่ที่: {output_dir}")
    print("=========================================")

In [ ]:
part_sec = 'part_1'
max_workers_training = 5
api_key_gemini = ''

my_prompt = """
คุณคือผู้เชี่ยวชาญด้าน Vision AI และการสกัดข้อมูลจากภาพเอกสารรายงานผลการนับคะแนนเลือกตั้งที่มีความแม่นยำสูงที่สุด
หน้าที่ของคุณคืออ่านข้อมูลจากภาพตารางคะแนนที่แนบมา และแปลงเป็นรูปแบบ JSON เพื่อนำไปอัปเดตข้อมูลในระบบฐานข้อมูล

ข้อมูลสำคัญที่ต้องสกัดในแต่ละแถว:
1. คะแนนตัวเลข (ประเภทตัวเลข ไทย, อารบิก เอาตามภาพเลย)
2. คำอ่านคะแนน (ตัวหนังสือภาษาไทย)

กฎเกณฑ์การสกัดข้อมูล (บังคับใช้เด็ดขาด):
1. ช่องใดที่ไม่มีคะแนน, ถูกขีดแดช (-), หรือปล่อยว่าง ให้กำหนด score_numeric เป็น "0" และ score_text เป็น "ศูนย์"
2. การอ่านคะแนนรวม (Page Validation): ให้ค้นหาช่อง "รวมคะแนน" หรือบรรทัดสรุปผลรวมที่ด้านล่างสุดของตาราง แล้วดึงตัวเลขและคำอ่านมาใส่ใน "page_summary" หากหาไม่เจอให้ใส่ null
3. หากตัวเลขและตัวหนังสือขัดแย้งกัน หรืออ่านยาก ให้อ่านตามสิ่งที่เห็นชัดเจนที่สุดในภาพ
4. ห้ามอธิบายใดๆ เพิ่มเติมเด็ดขาด ให้ตอบกลับมาเป็น JSON ตามโครงสร้างด้านล่างนี้เท่านั้น ห้ามใช้ Markdown code block (```json ... ```) คลุม หากไม่จำเป็น

โครงสร้าง JSON ที่ต้องการ:
{
  "results": [
    {
      "score_numeric": "154",
      "score_text": "หนึ่งร้อยห้าสิบสี่"
    },
    {
      "score_numeric": "๒๐",
      "score_text": "ยี่สิบ"
    }
  ],
  "page_summary": {
    "total_score_numeric": "1050",
    "total_score_text": "หนึ่งพันห้าสิบ"
  }
}
"""

input_folder = '/content/images/'
input_folder_after_manual = '/content/DataCleaned_m/content/output_image_cleaned_folder'
input_json_final = '/content/output_json_final'
input_template_csv_path = '/content/submission_template_v4.csv'

output_image_cleaned_folder = 'output_image_cleaned_folder'
output_image_cleaned_noChunk_folder = 'output_image_cleaned_noChunk_folder'
output_json_after_ocr = 'output_json_after_ocr'
output_json_after_ocr_2 = 'output_json_after_ocr_2'
output_json_after_ocr_merged_json = 'output_json_after_ocr_merged_json'

output_json_final = 'output_json_final'
output_csv_path = '/content/submission.csv'

In [ ]:
preProcessing(input_folder, output_image_cleaned_folder, output_image_cleaned_noChunk_folder)

In [ ]:
!zip -r DataCleaned.zip /content/output_image_cleaned_folder/

In [ ]:
files.download('DataCleaned.zip')

# ไปดู Data ด้วยมือ ละคลีนเองอีกทีนีง(manual) แล้วอัพลงไดฟ์ตั้งชื่อ `DataCleaned_m`

In [ ]:
!unzip /content/drive/MyDrive/SUPER_AI/DataCleaned_m.zip

In [ ]:
model_training(input_folder_after_manual, output_json_after_ocr)

In [ ]:
!zip -r submissionJsonFile.zip /content/output_json_after_ocr

In [ ]:
files.download('submissionJsonFile.zip')

# ทำ Post-process
- merge .json ที่ถูกหั่น มารวมไว้ในไฟล์เดียวกัน
- แปลงตัวเลขไทย มาเป็น อารบิก, ตัวอักษร มาเป็น อารบิก
- หาผลรวมด้วยตัวเลข ตรงมั้ย ถ้าไม่ตรง
- หาผลรวมด้วยตัวอักษร ตรงมั้ย ถ้าไม่ตรง
- แยกมาไว้อีกทีนึง (passed_complete, needs_review)
    - ถ้าไม่เหมือน (needs_review) ให้ยึดตาม ตัวอักษร ของภาพ
- พอได้ final json file มาแล้วเนี่ยทำการ map to csv

In [ ]:
merged_json(output_json_after_ocr, output_json_after_ocr_merged_json)

In [ ]:
!zip -r output_json_after_ocr_merged_json.zip /content/output_json_after_ocr_merged_json

In [ ]:
files.download('output_json_after_ocr_merged_json.zip')

In [ ]:
forCheck = validate_and_categorize_results(output_json_after_ocr_merged_json)

In [ ]:
print(forCheck)

In [ ]:
if len(forCheck) > 0:
    copy_images_for_recheck(forCheck, ALL_IMAGES_DIR, RECHECK_DIR)
    print("พร้อมรีเช็คแ้ว")
else:
    print("ไม่มีไฟล์ไหนต้องรีเช็คเลย")

In [ ]:
generate_final_score_json(output_json_after_ocr_merged_json, forCheck, output_json_final)

In [ ]:
!zip -r output_json_final.zip /content/output_json_final

In [ ]:
files.download('output_json_final.zip')

In [ ]:
merge_scores_to_submission(input_json_final, input_template_csv_path, output_csv_path)

In [47]:
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>